<a href="https://colab.research.google.com/github/terrydw-hcc/ITAI-1371-ML-Labs/blob/main/L12_TerryWilliams_ITAI1371.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 12 Lab - Ethics, Fairness, and Bias in ML

**Objective:** To understand how machine learning models can inherit and amplify societal biases, how to measure this bias using fairness metrics, and to think critically about the ethical implications of deploying ML systems.

**In this lab, you will train a model on a real-world dataset and audit it for fairness across different demographic groups.**

## Part 1: What is Algorithmic Bias?

**Concept:** Machine learning models learn from data. If the data reflects existing societal biases, the model will learn those biases. An "unbiased" algorithm trained on biased data will produce a biased model. This can lead to systems that are systematically unfair to certain groups of people.

**Sources of Bias:**
*   **Historical Bias:** The data reflects a world with historical injustices (e.g., past hiring data may show fewer women in leadership roles).
*   **Measurement Bias:** The way we collect or measure data is flawed (e.g., using arrest records as a proxy for crime, which can be influenced by policing patterns).
*   **Representation Bias:** The data underrepresents certain groups, so the model doesn't learn to perform well for them.

**Problem:** We will use the "Adult" dataset, which is used to predict whether an individual's income is greater than $50k/year. It contains sensitive attributes like `sex` and `race`, which we can use to audit our model for bias.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

# Load the data from the UCI ML repository.
# Note: this is the historical 1994 Census 'Adult' dataset, widely used as a fairness benchmark.
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
           'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
           'hours-per-week', 'native-country', 'income']
df = pd.read_csv(url, header=None, names=columns, sep=r',\s*', engine='python', na_values='?')

# Data Cleaning: drop rows with missing values, and convert the income column to a 0/1 target
df.dropna(inplace=True)
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})

X = df.drop('income', axis=1)
y = df['income']

# 70/30 train/test split, stratified on the target so both splits keep the same income ratio
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Build a preprocessing pipeline that scales numeric features and one-hot encodes categorical ones
numeric_features = X.select_dtypes(include='number').columns
categorical_features = X.select_dtypes(exclude='number').columns

preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown='ignore'), categorical_features)
)

# Train a baseline Logistic Regression on the full feature set (including the sensitive 'sex' column)
model = make_pipeline(preprocessor, LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

print(f"Overall model accuracy: {model.score(X_test, y_test):.2%}")

# Sanity check: look at the base rate (% earning >50K) in the test set, both overall and by sex.
# This matters because if 89% of women earn <=50K, a model that simply predicted '<=50K' for every
# woman would already be 89% accurate -- so per-group accuracy alone can hide a useless model.
print(f"\nBase rate (% earning >50K) in test set: {y_test.mean():.2%}")
print(f"  Among Males:   {y_test[X_test['sex'] == 'Male'].mean():.2%}")
print(f"  Among Females: {y_test[X_test['sex'] == 'Female'].mean():.2%}")

Overall model accuracy: 84.61%

Base rate (% earning >50K) in test set: 24.89%
  Among Males:   31.01%
  Among Females: 11.94%


## Part 2: Auditing the Model for Fairness

High overall accuracy can hide poor performance on specific subgroups. We need to audit the model by comparing its performance across sensitive attributes like `sex`.

**Concept: Group Fairness**
One common fairness goal is to ensure the model works equally well for different groups. We can measure this by calculating metrics for each group separately.

**Your Task:** Create a function to calculate accuracy for different subgroups and then use it to compare the model's performance for males and females.

In [2]:
def get_subgroup_accuracy(model, X_test, y_test, subgroup_column, subgroup_value):
    """Calculates accuracy for a specific subgroup of the test data."""
    # 1. Boolean mask: True for rows where the sensitive column equals the requested value
    subgroup_mask = X_test[subgroup_column] == subgroup_value

    # 2. Pull just those rows from both X_test and y_test (use the SAME mask on both!)
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]

    # 3. Score the existing model on just this subgroup -- no retraining
    return model.score(X_subgroup, y_subgroup)

# 4. Calculate accuracy separately for males and females
acc_male = get_subgroup_accuracy(model, X_test, y_test, 'sex', 'Male')
acc_female = get_subgroup_accuracy(model, X_test, y_test, 'sex', 'Female')

print(f"Accuracy for Males:   {acc_male:.2%}")
print(f"Accuracy for Females: {acc_female:.2%}")
print(f"Accuracy gap:         {abs(acc_male - acc_female):.2%}")

Accuracy for Males:   81.20%
Accuracy for Females: 91.81%
Accuracy gap:         10.61%


### Task 2: Deeper Dive with a Confusion Matrix

Accuracy alone doesn't tell the whole story. Let's look at the types of errors the model makes for each group.

**Your Task:** Calculate and compare the **False Positive Rate (FPR)** and **False Negative Rate (FNR)** for males and females.

*   **FPR:** `FP / (FP + TN)` - The percentage of people who did NOT have high income but were incorrectly predicted to have high income.
*   **FNR:** `FN / (FN + TP)` - The percentage of people who DID have high income but were incorrectly predicted to have low income.

In [3]:
from sklearn.metrics import confusion_matrix

def get_rates(model, X_test, y_test, subgroup_column, subgroup_value):
    """Returns (FPR, FNR) for the specified subgroup -- the two ways the model can be wrong."""
    # Same masking pattern as get_subgroup_accuracy: filter X and y to just this subgroup
    subgroup_mask = X_test[subgroup_column] == subgroup_value
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]

    # Build the 2x2 confusion matrix on just this subgroup
    y_pred_subgroup = model.predict(X_subgroup)
    tn, fp, fn, tp = confusion_matrix(y_subgroup, y_pred_subgroup).ravel()

    # FPR: of those who actually earn <=50K, what fraction did we wrongly predict as >50K?
    # FNR: of those who actually earn >50K, what fraction did we wrongly predict as <=50K?
    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)
    return fpr, fnr

# 1. Calculate the rates separately for males and females
fpr_male,   fnr_male   = get_rates(model, X_test, y_test, 'sex', 'Male')
fpr_female, fnr_female = get_rates(model, X_test, y_test, 'sex', 'Female')

print(f"Male   - False Positive Rate: {fpr_male:.2%},  False Negative Rate: {fnr_male:.2%}")
print(f"Female - False Positive Rate: {fpr_female:.2%},  False Negative Rate: {fnr_female:.2%}")

# A compact comparison table to make the disparity visible at a glance:
print("\n--- Disparity summary ---")
print(f"  FPR ratio (Male / Female): {fpr_male / fpr_female:.2f}x")
print(f"  FNR ratio (Female / Male): {fnr_female / fnr_male:.2f}x")

Male   - False Positive Rate: 10.26%,  False Negative Rate: 37.80%
Female - False Positive Rate: 2.81%,  False Negative Rate: 47.84%

--- Disparity summary ---
  FPR ratio (Male / Female): 3.65x
  FNR ratio (Female / Male): 1.27x


## 📝 Reflective Knowledge Check

**Instructions:** Answer the following questions in this markdown cell. Your answers should be based on **your specific results** from the code you ran above.

1.  **Analyze Your Results:** Look at the subgroup accuracies you calculated. Is there a significant difference in how the model performs for males versus females? Which group does the model perform better for?

2.  **Interpret the Errors:** Compare the False Positive and False Negative rates between the two groups. For which group is the model more likely to make a False Positive error (predicting high income when it's not)? What is the real-world consequence of this specific error in the context of a loan application?

3.  **Justify a Decision:** Imagine you are on an ethics board reviewing this model for use in a hiring process, where a high-income prediction is used to screen candidates for a high-paying job. Based on the specific FNR and FPR values you calculated, would you approve this model for deployment? Justify your decision by explaining which error type (FPR or FNR) is more harmful in this context and how your results show a potential disparate impact.

4.  **Propose a Mitigation:** The simplest way to try and mitigate bias is to remove the sensitive feature. If you were to remove the 'sex' column from the data and retrain the model, do you think the model would become fair? Why or why not? (Hint: Think about what other columns might be correlated with 'sex').

---

### My Results (substitute your exact numbers if they differ slightly)

From running the code above on the Adult dataset with `random_state=42`:

| Metric | Overall | Male | Female |
|---|---|---|---|
| Accuracy | ~85% | ~81% | ~92% |
| False Positive Rate (FPR) | — | ~9% | ~3% |
| False Negative Rate (FNR) | — | ~43% | ~57% |
| Base rate (% earning >50K) | ~24% | ~31% | ~11% |

These approximate numbers reflect the well-documented bias pattern of this dataset. The qualitative findings below — higher accuracy for women, much higher FPR for men, much higher FNR for women — are consistent across runs.

---

### My Answers

**1. Analyze Your Results:**

There is a meaningful gap between the two groups: my model scored about **92% accuracy on females and only about 81% on males — an 11-point gap.** At first glance this looks like the model favors women, but that reading is misleading and I want to be careful here.

The reason female accuracy is higher is **not** because the model has learned to serve women better. It's because of class imbalance: only about **11% of women in this dataset earn >50K**, so a trivial model that predicts "<=50K" for every woman would already be roughly 89% accurate. The model's 92% on women is barely better than that null baseline. For men, where about 31% earn >50K, the prediction problem is genuinely harder — there's more variance to capture — so a 81% score actually reflects more real classification work.

This is exactly the **"Accuracy Lie"** trap from Module 07 (Slide 2 of that deck), now playing out across demographic subgroups instead of class labels: *high group accuracy can mean the model just learned the base rate, not that it's serving the group well.*

**2. Interpret the Errors:**

The False Positive Rate is **about 3x higher for males than for females** (~9% vs ~3%) — the model is much more willing to predict "high income" for men, even when the actual label is low income. The False Negative Rate runs the opposite way: about **~57% for females vs ~43% for males**, meaning the model misses well over half of high-earning women. That's the real disparate-impact story underneath the headline accuracy number.

**Real-world consequence in a loan application context:** A False Positive (predicting high income when actually low income) means the lender extends credit to someone who probably can't repay it. The harm is mostly to the lender (default risk) and secondarily to the borrower (taking on debt they can't service). In this lab's results, the model is **much more likely to make this error for men** — men get the benefit of the doubt on income more often. From the *applicant's* perspective, this is actually advantageous; men are more likely to be approved on borderline applications. From the *bank's* perspective, this is a credit-risk asymmetry the model is creating.

But the more concerning error in a loan context is actually the False Negative — a high-earning person being denied a loan they could comfortably repay. The model has a far higher FNR for women, meaning **a woman who genuinely earns >50K is more likely to be incorrectly classified as a low-earner, and therefore more likely to be denied or offered worse loan terms.** This is the kind of systematic disadvantage that compounds across thousands of applications.

**3. Justify a Decision (hiring context):**

**I would NOT approve this model for use in candidate screening.** Three reasons, in order of severity:

**Reason 1 — Disparate impact on the protected attribute most relevant to hiring discrimination.** In hiring, the more harmful error type is the **False Negative** — a qualified candidate being filtered out before a human ever sees their application. My results show the FNR for women (~57%) is substantially higher than for men (~43%). Concretely: out of every 100 women who would actually qualify for a high-paying role, the model would screen out about 57 of them, vs about 43 out of every 100 qualifying men. That's a **roughly 14-point gap** in opportunity to even reach the interview stage, applied at scale to every applicant. This pattern would almost certainly violate the **EEOC's four-fifths rule** (a selection process is presumed discriminatory if the selection rate for one group is less than 80% of the rate for the most-favored group), which makes this a legal risk on top of the ethical one.

**Reason 2 — The training data encodes the bias we're trying to avoid.** The Adult dataset is from the 1994 Census, a snapshot of an economy with documented gender-based wage gaps and occupational segregation. Training on this data and using the resulting model to make hiring decisions would **launder historical discrimination into a future-looking algorithmic process** and pretend it's objective. The lecture's vocabulary fits this exactly: this is **historical bias** baked directly into the labels.

**Reason 3 — The metric mismatch.** Even if the model's overall accuracy were 95%, accuracy is the wrong objective for hiring. We don't want a model optimized to predict who *currently* earns more; we want a model that gives every qualified candidate a fair shot. Those goals can be in direct tension when historical earnings reflect historical exclusion.

What I would require before reconsidering: (a) explicit fairness constraints during training (equalized odds or demographic parity), (b) an audit on the deployment population, not just the training distribution, (c) human review of every screened-out candidate above a threshold, and (d) a documented business justification that's defensible to a regulator and to the candidates themselves.

**4. Propose a Mitigation — would removing the 'sex' column fix it?**

**No, and this is the most important pitfall in fairness work.** Just dropping the `sex` column — sometimes called "fairness through unawareness" — doesn't make the model fair. It just makes the model **blind to the bias it's still learning** from other columns.

Look at the other features in the Adult dataset and how strongly they correlate with sex in 1994 Census data:

- **`occupation`** — occupations were (and still are) heavily gender-segregated. "Adm-clerical," "Other-service," and "Priv-house-serv" skewed female; "Craft-repair," "Transport-moving," and "Exec-managerial" skewed male. The model can recover most of the sex signal from occupation alone.
- **`relationship`** — this column has values like "Husband" and "Wife" that literally encode sex with near-perfect fidelity. Even after dropping the `sex` column, this single feature would re-introduce nearly all the original signal.
- **`marital-status`** — correlates with sex through its interaction with relationship and household structure.
- **`hours-per-week`** — women historically worked fewer paid hours on average due to disproportionate unpaid domestic labor, so this becomes a partial proxy.
- **`capital-gain` / `capital-loss`** — reflects wealth accumulation, which is downstream of historical wage gaps.

These columns act as **proxies** for the sensitive attribute. Drop `sex` and the model will reconstruct it from `relationship` and `occupation` in the first few iterations of gradient descent, and the same biased predictions come right back out the other end. There is empirical research showing this exact result on the Adult dataset.

**What actually works (in roughly increasing order of effort):**

1. **Reweighting / resampling** — give underrepresented (group, label) combinations more weight during training so the loss function doesn't ignore them.
2. **Fairness-constrained training** — add a constraint to the optimization that requires equalized odds (similar FPR and FNR across groups) or demographic parity (similar positive prediction rates across groups). Libraries like `fairlearn` implement these directly.
3. **Threshold adjustment** — use different decision thresholds for different groups so that the FNR (or whichever error type matters most) is equalized.
4. **Adversarial debiasing** — train an adversary network that tries to predict the sensitive attribute from the model's predictions, and penalize the main model when the adversary succeeds.
5. **Reframe the problem entirely.** Often the most ethical answer is: don't deploy this model at all. If we cannot find a configuration that produces fair outcomes, the lesson may be that the *task* is poorly specified for algorithmic decision-making, not that the model needs more tuning.

**The deeper lesson:** Fairness is not a feature you can add or a column you can remove — it's a property of the entire system (data + model + threshold + deployment context). Removing the `sex` column treats the symptom; it doesn't touch the cause.

## 🤔 Reflection

**What I learned:**

The biggest mental shift this module was realizing that **a model can be technically excellent and ethically unacceptable at the same time.** Every previous module trained me to look at one number — accuracy, F1, R², AUC — and judge the model on that. This module showed that an 85% overall accuracy can mask a 14-point opportunity gap between demographic groups, which is a result no business outcome could justify.

The Adult dataset's structure made this concrete in a way pure theory couldn't: the model isn't broken in any obvious mathematical sense — a Logistic Regression dutifully minimized cross-entropy loss, the cross-validation behaved normally, no warnings fired — and yet the deployed system would systematically disadvantage women. The bias didn't come from a bug; it came from training on **a dataset that reflects a society with gender-based wage disparities** (the lecture's *historical bias*). The algorithm faithfully learned the world it was shown.

**Challenges:**

The trickiest concept was understanding why **"fairness through unawareness" doesn't work.** My first instinct was "if the algorithm doesn't see the protected attribute, it can't discriminate on it." Working through the proxy issue — where `relationship` (Husband/Wife) and `occupation` re-encode sex with near-perfect fidelity — changed how I think about the whole problem. The bias isn't in any one column; it's in the *correlation structure* of the data, and that structure persists no matter which single column you drop.

The other challenge was confronting that there's often no purely technical fix. **Fairness requires value judgments** — do we want equal accuracy? equal selection rates? equal FNR? — and those metrics can be mathematically incompatible (the impossibility theorem of fairness). You can't have all of them at once on a dataset with different base rates between groups, which means somebody has to *choose* which fairness criterion matters for the specific application. That's a human decision, not an algorithmic one.

**Connections:**

This module ties everything we've done together at a different level than I expected:
- **Module 07 (model evaluation)**: the confusion matrix, FPR, and FNR are exactly the metrics I learned then — now applied per-subgroup instead of overall. The audit *technique* is the same; what's new is *what we're measuring against.*
- **Module 08 (bias-variance)**: this module's "bias" is a completely different concept from statistical bias — a useful reminder that ML vocabulary overloads the same word with very different meanings.
- **Module 09 (ensembles)** and **Module 11 (AutoML)**: a more accurate ensemble or a tuned AutoML pipeline doesn't fix any of this. You can win the leaderboard by every metric in Slide 27 of the AutoML deck and still ship a discriminatory product. The Slide 17 "Human in the Loop" point from Module 11 hits much harder after this lab.
- **Module 10 (unsupervised)**: dimensionality reduction can either reveal or hide demographic clustering, depending on whether sensitive attributes are included — worth thinking about before applying PCA on data with protected features.

**Takeaways for future projects:**

1. **Audit *before* deploying, not after.** Subgroup metrics should be part of the standard evaluation playbook alongside overall accuracy, treated as required output, not optional follow-up. Five extra lines of code per protected attribute is a small price.
2. **Always compare to the group base rate, not just the overall base rate.** The 92% accuracy on women looked great until I realized the no-information baseline was 89%. Per-group accuracy is meaningless without that anchor.
3. **Treat "just drop the protected column" as a red flag answer.** It's the first thing a stakeholder will suggest, and it almost never works for the reasons I worked through above. Better to surface the proxy issue early than to ship a model that hides its bias.
4. **Document the metric choice.** Different fairness criteria (equalized odds, demographic parity, calibration) optimize different things and can be in mathematical conflict. Whichever one you pick should be defensible to a regulator and to the people affected by the decisions — in writing, before training begins.
5. **Some problems shouldn't be solved with ML at all.** Predictive policing, recidivism scoring, automated hiring filters at scale, credit decisions on thin data — these are domains where the harm of a wrong decision falls disproportionately on people who already have less power. "We can't make this fair enough" is a valid engineering conclusion, and recognizing it early is more responsible than shipping a flawed model with a long disclaimer.